# **Qwen2.5-VL Car Goes Around MuJoCo Circuit**

### **Ci: Colab L4 2026/05/11 12:00**

In [ ]:
!apt-get install -y libglew-dev libegl1-mesa-dev libgles2-mesa-dev
!pip install mediapy mujoco torch imageio
!pip install qwen-vl-utils

    # ============================================================
    # Qwen2.5-VL Car Goes Around MuJoCo Circuit  —  Vision Only
    #
    #   No track geometry pre-knowledge.
    #   No centerline waypoints. No FF. No FB.
    #
    #   Steering = Qwen visual judgment only.
    #   Wall distances = MuJoCo raycasting (sensor, not map).
    #
    #   Qwen now judges TWO things from the image:
    #     1. curve direction  → steer_right / steer_left / none
    #     2. lateral drift    → drifting_left / centered / drifting_right
    #
    #   Based on:
    #     Qwen2.5-VL Car Goes Around MuJoCo Circuit wo/SC
    #     https://www.kaggle.com/code/stpeteishii/car-goes-around-mujoco-circuit2
    # ============================================================

In [ ]:
import os
os.environ['MUJOCO_GL']         = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

import json, re, base64, math
import xml.etree.ElementTree as ET
from xml.dom import minidom
from io import BytesIO

import mujoco, imageio
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [ ]:
# ============================================================
# Parameters
# ============================================================

USE_QWEN = True

SIM_DT            = 0.01
SIM_STEPS         = 800
RECORD_EVERY      = 4
FPS_OUTPUT        = 30
MAX_SIM_TIME      = 10.0
STUCK_DIST_THRESH = 0.5
STUCK_WINDOW      = 0.5

# Track Dimensions
SL        = 12.0
R_CENTER  = 30.0
LANE_W    =  6.0
R_INNER   = R_CENTER - LANE_W / 2
R_OUTER   = R_CENTER + LANE_W / 2
WALL_T    =  0.5
WALL_H    =  1.3
N_CURVE   = 28

# Vehicle Specs
WHEEL_RADIUS  = 0.30
WHEELBASE     = 2.40
TRACK_WIDTH   = 1.20
CHASSIS_H     = 0.35
MAX_STEER_RAD = 0.45

CAR_BODY_MASS  = 800.0
CAR_CABIN_MASS =  80.0
CAR_WHEEL_MASS =  15.0

X0, Y0, H0 = 0.0, R_CENTER, 0.0
TARGET_V   = 18 * 1000 / 3600   # 18 km/h → m/s

JUDGMENT_INTERVAL = 1
STEER_LERP        = 0.15
STEER_DECAY       = 0.995

SAFETY_DIST       = 1.5
SAFETY_STEER_MAG  = 0.95

# ── ★ Vision-Only Steering Gains ─────────────────────────────
# All steering magnitude is determined by these values +
# Qwen's confidence. No geometric gains (K_FF / K_HEADING /
# LATERAL_GAIN) exist in this version.

DRIFT_STEER_BASE = math.radians(8)   # base steer for drift correction
CURVE_STEER_BASE = math.radians(6)  
CURVE_STEER_MAX = math.radians(22) 
DRIFT_GAIN       = 0.05              

# ── Centerline Dashed Line Params ─────────────────────────────
CL_DASH_INTERVAL = 1.5
CL_DASH_LEN      = 0.70
CL_DASH_WIDTH    = 0.15
CL_DASH_H        = 0.012
CL_DASH_RGBA     = '1.0 0.95 0.0 1'

In [ ]:
# ============================================================
# Wall distance via MuJoCo raycasting
# — no track map required
# ============================================================
def get_wall_distances_raycast(model, data, car_pos, car_hdg, car_bid):
    ray_height = 0.50                        
    pos3 = np.array([car_pos[0], car_pos[1], ray_height], dtype=np.float64)

    perp_left  = np.array([-math.sin(car_hdg),  math.cos(car_hdg), 0.0])
    perp_right = np.array([ math.sin(car_hdg), -math.cos(car_hdg), 0.0])

    geomid_l = np.array([-1], dtype=np.int32)
    geomid_r = np.array([-1], dtype=np.int32)

    dist_l = mujoco.mj_ray(model, data, pos3, perp_left,
                            None, 1, car_bid, geomid_l)   # -1 → car_bid
    dist_r = mujoco.mj_ray(model, data, pos3, perp_right,
                            None, 1, car_bid, geomid_r)   # -1 → car_bid

    dist_l = float(dist_l) if dist_l > 0 else LANE_W
    dist_r = float(dist_r) if dist_r > 0 else LANE_W

    return min(dist_l, LANE_W), min(dist_r, LANE_W)

In [ ]:
# ============================================================
# Emergency wall override  (unchanged — uses raycast distances)
# ============================================================
def geo_safety_override(geo_left, geo_right):
    if geo_left >= SAFETY_DIST and geo_right >= SAFETY_DIST:
        return 0.0, False
    base = SAFETY_STEER_MAG * MAX_STEER_RAD
    if geo_left < SAFETY_DIST and geo_right < SAFETY_DIST:
        if geo_left >= geo_right:
            factor = 1.0 + (SAFETY_DIST - geo_right) / SAFETY_DIST
            return +float(np.clip(factor * base, 0, MAX_STEER_RAD)), True
        else:
            factor = 1.0 + (SAFETY_DIST - geo_left) / SAFETY_DIST
            return -float(np.clip(factor * base, 0, MAX_STEER_RAD)), True
    elif geo_left < SAFETY_DIST:
        factor = 1.0 + (SAFETY_DIST - geo_left) / SAFETY_DIST
        return -float(np.clip(factor * base, 0, MAX_STEER_RAD)), True
    else:
        factor = 1.0 + (SAFETY_DIST - geo_right) / SAFETY_DIST
        return +float(np.clip(factor * base, 0, MAX_STEER_RAD)), True

In [ ]:
# ============================================================
# Vision-only steering target
# ============================================================
def compute_steer_target_vision_only(curve_action, curve_intensity,
                                     curve_conf, geo_left, geo_right,
                                     current_target):
    steer = 0.0

    if curve_action == "steer_right":
        steer -= CURVE_STEER_MAX * curve_intensity * curve_conf
    elif curve_action == "steer_left":
        steer += CURVE_STEER_MAX * curve_intensity * curve_conf
    else:
        steer = current_target * STEER_DECAY

    lateral_imbalance = geo_right - geo_left
    steer += DRIFT_GAIN * lateral_imbalance

    return float(np.clip(steer, -MAX_STEER_RAD, MAX_STEER_RAD))

In [ ]:
# ============================================================
# HUD  — shows raycast distances + Qwen judgments
# ============================================================

def draw_hud(pil_img, geo_left, geo_right, qwen_curve, qwen_drift,
             curve_conf, drift_conf):
    img  = pil_img.copy()
    draw = ImageDraw.Draw(img)
    W, H = img.size

    def wall_col(d):
        if d < SAFETY_DIST:        return (255, 60, 60)
        if d < SAFETY_DIST + 1.0:  return (255, 220, 50)
        return (80, 255, 80)

    try:
        fL = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 19)
        fS = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 14)
    except Exception:
        fL = fS = ImageFont.load_default()

    draw.rectangle([4, 4, 240, 115], fill=(0, 0, 0, 180))
    draw.text((10,  6), f"L={geo_left:.2f}m",  font=fL, fill=wall_col(geo_left))
    draw.text((10, 30), f"R={geo_right:.2f}m", font=fS, fill=wall_col(geo_right))
    draw.text((10, 52), f"curve : {qwen_curve}",
              font=fS, fill=(180, 220, 255))
    draw.text((10, 70), f"  conf: {curve_conf:.2f}",
              font=fS, fill=(180, 220, 255))
    draw.text((10, 88), f"drift : {qwen_drift}",
              font=fS, fill=(200, 255, 180))
    draw.text((10,106), f"  conf: {drift_conf:.2f}",
              font=fS, fill=(200, 255, 180))

    return img

In [ ]:
# ============================================================
# Qwen2.5-VL  — expanded prompt: curve + drift
# ============================================================

print("Loading Qwen2.5-VL...")
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto")
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Model loaded.")

QWEN_PROMPT = """\
You are the vision system of a toy car driving on a clockwise oval circuit.

On the track you can see a bright YELLOW dashed center line.

Judge TWO things from the image:

1. CURVE — Which way does the yellow line curve AHEAD?
   Right curve  → curve_action: "steer_right"
   Left curve   → curve_action: "steer_left"
   Straight     → curve_action: "none"

2. DRIFT — Is the car currently centered on the yellow line,
   or has it drifted to one side?
   Car is to the LEFT  of center → drift_action: "drifting_left"
   Car is to the RIGHT of center → drift_action: "drifting_right"
   Car is on center              → drift_action: "centered"

Base BOTH judgments solely on what you see in the image.
The HUD numbers (L=, R=) show sensor distances to the walls — use them
as supporting context for the drift judgment if helpful.

Output ONE JSON line only — no prose, no markdown:
{
  "curve_direction": "<STRAIGHT|CURVE_L|CURVE_R>",
  "curve_action":    "<none|steer_left|steer_right>",
  "curve_intensity": 0.0-1.0, 
  "curve_conf":      0.0-1.0,
  "drift_action":    "<centered|drifting_left|drifting_right>",
  "drift_conf":      0.0-1.0
}


"""

In [ ]:
def query_qwen(image: Image.Image) -> str:
    buf = BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    messages = [{
        "role": "user",
        "content": [
            {"type": "text",  "text": QWEN_PROMPT},
            {"type": "image", "image": f"data:image/png;base64,{b64}"},
            {"type": "text",  "text": "JSON only:"},
        ],
    }]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(**inputs, max_new_tokens=96)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return qwen_processor.batch_decode(trimmed, skip_special_tokens=True)[0]


def parse_response(s: str) -> tuple:
    """Returns (curve_action, drift_action, curve_conf, drift_conf)."""
    try:
        clean = re.sub(r'```[a-z]*', '', s).strip()
        m = re.search(r'\{[^{}]*\}', clean, re.DOTALL)
        if not m:
            return "none", "centered", 0.0, 0.0
        p           = json.loads(m.group(0))
        curve_act   = str(p.get("curve_action",  "none")).lower().strip()
        drift_act   = str(p.get("drift_action",  "centered")).lower().strip()
        curve_conf  = float(p.get("curve_conf",  0.5))
        drift_conf  = float(p.get("drift_conf",  0.5))
        return curve_act, drift_act, curve_conf, drift_conf
    except Exception as e:
        print(f"  [PARSE ERROR] {e}")
        return "none", "centered", 0.0, 0.0

In [ ]:
# ============================================================
# Centerline dashes  — visual only, NOT used for control
# ============================================================

def make_centerline_visual(n: int = 100):
    """
    Used only to render the yellow dashes Qwen will look at.
    Never referenced by the steering controller.
    """
    R, hs = R_CENTER, SL / 2
    pts, hdgs = [], []
    for x in np.linspace(-hs, hs, n):
        pts.append([x, R]);   hdgs.append(0.0)
    for th in np.linspace(np.pi / 2, -np.pi / 2, n):
        pts.append([hs + R * np.cos(th), R * np.sin(th)])
        hdgs.append(np.arctan2(-np.cos(th), np.sin(th)))
    for x in np.linspace(hs, -hs, n):
        pts.append([x, -R]);  hdgs.append(np.pi)
    for th in np.linspace(-np.pi / 2, -3 * np.pi / 2, n):
        pts.append([-hs + R * np.cos(th), R * np.sin(th)])
        hdgs.append(np.arctan2(-np.cos(th), np.sin(th)))
    return np.array(pts), np.array(hdgs)

In [ ]:
# ============================================================
# XML builder  (identical to original)
# ============================================================

def yaw_to_quat(yaw):
    return np.cos(yaw / 2), 0.0, 0.0, np.sin(yaw / 2)


def build_track(parent):
    tbody = ET.SubElement(parent, 'body', attrib={'name': 'track', 'pos': '0 0 0'})
    hw, ht, hs = WALL_H / 2, WALL_T / 2, SL / 2
    c_out, c_in = '0.85 0.3 0.2 1', '0.2 0.5 0.85 1'

    def wall_box(name, px, py, hl, rgba, ez=0.0):
        ET.SubElement(tbody, 'geom', attrib={
            'name': name, 'type': 'box',
            'size': f'{hl:.4f} {ht:.4f} {hw:.4f}',
            'pos':  f'{px:.4f} {py:.4f} {hw:.4f}',
            'euler': f'0 0 {ez:.4f}', 'rgba': rgba,
            'contype': '1', 'conaffinity': '1',
        })

    wall_box('sw_top_out', 0.0,  R_OUTER + ht, hs, c_out)
    wall_box('sw_bot_out', 0.0, -R_OUTER - ht, hs, c_out)
    wall_box('sw_top_in',  0.0,  R_INNER - ht, hs, c_in)
    wall_box('sw_bot_in',  0.0, -R_INNER + ht, hs, c_in)

    dth = np.pi / N_CURVE
    for side, cx, a0, a1 in [
        ('r', +hs, -np.pi/2, +np.pi/2),
        ('l', -hs, +np.pi/2, +3*np.pi/2),
    ]:
        for i, th in enumerate(np.linspace(a0 + dth/2, a1 - dth/2, N_CURVE)):
            ez = th + np.pi / 2
            wall_box(f'cw_{side}_{i}_out',
                     cx + R_OUTER*np.cos(th), R_OUTER*np.sin(th),
                     2*R_OUTER*np.sin(dth/2)/2 + 0.03, c_out, ez)
            wall_box(f'cw_{side}_{i}_in',
                     cx + R_INNER*np.cos(th), R_INNER*np.sin(th),
                     2*R_INNER*np.sin(dth/2)/2 + 0.03, c_in, ez)


def build_centerline_dashes(parent, cl_pts, cl_hdgs):
    tbody  = ET.SubElement(parent, 'body',
                           attrib={'name': 'centerline_body', 'pos': '0 0 0'})
    total  = 2 * SL + 2 * np.pi * R_CENTER
    step   = max(1, int(CL_DASH_INTERVAL * len(cl_pts) / total))
    hl, hw2, hh = CL_DASH_LEN/2, CL_DASH_WIDTH/2, CL_DASH_H/2
    count  = 0
    for i in range(0, len(cl_pts), step):
        px, py = cl_pts[i]
        ET.SubElement(tbody, 'geom', attrib={
            'name':        f'cl_dash_{count}',
            'type':        'box',
            'size':        f'{hl:.4f} {hw2:.4f} {hh:.4f}',
            'pos':         f'{px:.4f} {py:.4f} {CL_DASH_H:.4f}',
            'euler':       f'0 0 {cl_hdgs[i]:.4f}',
            'rgba':        CL_DASH_RGBA,
            'contype':     '0', 'conaffinity': '0',
        })
        count += 1
    print(f'[CL dashes] {count} geoms  (visual only — not used for control)')


def build_car(parent, prefix, xy, heading, rgba,
              m_body=800., m_cabin=80., m_wheel=15.):
    qw, qx, qy, qz = yaw_to_quat(heading)
    x, y = xy
    chassis = ET.SubElement(parent, 'body', attrib={
        'name': f'{prefix}_chassis',
        'pos':  f'{x} {y} {CHASSIS_H}',
        'quat': f'{qw} {qx} {qy} {qz}',
    })
    ET.SubElement(chassis, 'freejoint', attrib={'name': f'{prefix}_free'})
    ET.SubElement(chassis, 'site',
                  attrib={'name': f'{prefix}_imu', 'pos': '0 0 0.1', 'size': '0.02'})
    for nm, sz, ps, col, ms in [
        (f'{prefix}_body',     '0.95 0.48 0.14', '0 0 0',         rgba,             str(m_body)),
        (f'{prefix}_cabin',    '0.52 0.42 0.14', '-0.08 0 0.28',  rgba,             str(m_cabin)),
        (f'{prefix}_bumper_f', '0.06 0.45 0.08', '0.98 0 -0.04',  '0.1 0.1 0.1 1', '15'),
        (f'{prefix}_bumper_r', '0.06 0.45 0.08', '-0.98 0 -0.04', '0.1 0.1 0.1 1', '15'),
    ]:
        ET.SubElement(chassis, 'geom', attrib={
            'name': nm, 'type': 'box', 'size': sz, 'pos': ps,
            'rgba': col, 'mass': ms, 'contype': '1', 'conaffinity': '1',
        })
    wb2, tw2 = WHEELBASE/2, TRACK_WIDTH/2
    wz = -(CHASSIS_H - WHEEL_RADIUS)
    for tag, wpos, has_steer in [
        ('fl', f'{wb2} {tw2} {wz:.3f}',   True),
        ('fr', f'{wb2} {-tw2} {wz:.3f}',  True),
        ('rl', f'{-wb2} {tw2} {wz:.3f}',  False),
        ('rr', f'{-wb2} {-tw2} {wz:.3f}', False),
    ]:
        wname = f'{prefix}_w_{tag}'
        wbody = ET.SubElement(chassis, 'body', attrib={'name': wname, 'pos': wpos})
        if has_steer:
            ET.SubElement(wbody, 'joint', attrib={
                'name': f'{wname}_s', 'type': 'hinge', 'axis': '0 0 1',
                'range': f'{-MAX_STEER_RAD} {MAX_STEER_RAD}', 'damping': '30',
            })
        ET.SubElement(wbody, 'joint', attrib={
            'name': f'{wname}_r', 'type': 'hinge', 'axis': '0 1 0',
            'range': '-1e6 1e6', 'damping': '0.5',
        })
        ET.SubElement(wbody, 'geom', attrib={
            'name': f'{wname}_tire', 'type': 'cylinder',
            'size': f'{WHEEL_RADIUS} 0.11', 'euler': '1.5708 0 0',
            'rgba': '0.1 0.1 0.1 1', 'mass': str(m_wheel),
            'contype': '1', 'conaffinity': '1',
        })

In [ ]:
# ============================================================
# Collision detection
# ============================================================

CAR_GEOMS = {
    'car1_body','car1_cabin','car1_bumper_f','car1_bumper_r',
    'car1_w_fl_tire','car1_w_fr_tire','car1_w_rl_tire','car1_w_rr_tire',
}

def _wall_geom_set():
    s = {'sw_top_out','sw_bot_out','sw_top_in','sw_bot_in'}
    for sd in ('r','l'):
        for i in range(N_CURVE):
            s.add(f'cw_{sd}_{i}_out'); s.add(f'cw_{sd}_{i}_in')
    return s

WALL_GEOMS = _wall_geom_set()

def is_wall_collision(model, data):
    for i in range(data.ncon):
        c  = data.contact[i]
        g1 = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, c.geom1)
        g2 = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, c.geom2)
        if (g1 in CAR_GEOMS and g2 in WALL_GEOMS) or \
           (g2 in CAR_GEOMS and g1 in WALL_GEOMS):
            return True
    return False

In [ ]:
# ============================================================
# Build XML & load model
# ============================================================

cl_pts_visual, cl_hdgs_visual = make_centerline_visual(n=100)

root = ET.Element('mujoco', attrib={'model': 'qwen_vision_only_circuit'})
ET.SubElement(root, 'compiler', attrib={'angle': 'radian'})
ET.SubElement(root, 'option',   attrib={'gravity': '0 0 -9.81', 'timestep': str(SIM_DT)})
dflt = ET.SubElement(root, 'default')
ET.SubElement(dflt, 'geom', attrib={'friction': '1.5 0.5 0.0001', 'condim': '4'})
wb = ET.SubElement(root, 'worldbody')

ET.SubElement(wb, 'geom', attrib={
    'name':'floor','type':'plane','size':'100 100 0.1',
    'rgba':'0.3 0.45 0.3 1','contype':'1','conaffinity':'1'})
ET.SubElement(wb, 'geom', attrib={
    'name':'asphalt','type':'box',
    'size':f'{SL/2+R_OUTER:.1f} {R_OUTER:.1f} 0.005',
    'pos':'0 0 0.005','rgba':'0.22 0.22 0.25 1',
    'contype':'0','conaffinity':'0'})
ET.SubElement(wb, 'light', attrib={
    'directional':'true','diffuse':'0.85 0.85 0.85','specular':'0.2 0.2 0.2',
    'pos':'0 0 30','dir':'0 0 -1','castshadow':'false'})
ET.SubElement(wb, 'light', attrib={
    'directional':'true','diffuse':'0.4 0.4 0.4','specular':'0 0 0',
    'pos':'20 -10 20','dir':'-1 0.5 -1','castshadow':'false'})

build_centerline_dashes(wb, cl_pts_visual, cl_hdgs_visual)
build_track(wb)
build_car(wb, 'car1', (X0, Y0), H0, rgba='0.9 0.15 0.15 1',
          m_body=CAR_BODY_MASS, m_cabin=CAR_CABIN_MASS, m_wheel=CAR_WHEEL_MASS)

act = ET.SubElement(root, 'actuator')
for nm, jt, kv in [('drv_rl','car1_w_rl_r','130'),('drv_rr','car1_w_rr_r','130')]:
    ET.SubElement(act, 'velocity', attrib={'name':nm,'joint':jt,'kv':kv})
for nm, jt, kp in [('str_fl','car1_w_fl_s','5000'),('str_fr','car1_w_fr_s','5000')]:
    ET.SubElement(act, 'position', attrib={'name':nm,'joint':jt,'kp':kp})

xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent='  ')
with open('qwen_vision_only_circuit.xml','w') as f:
    f.write(xml_str)
print('[XML] saved')

model  = mujoco.MjModel.from_xml_path('qwen_vision_only_circuit.xml')
data   = mujoco.MjData(model)
mujoco.mj_forward(model, data)

def _aid(n): return mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, n)
def _bid(n): return mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY,     n)

a_drl, a_drr = _aid('drv_rl'), _aid('drv_rr')
a_sfl, a_sfr = _aid('str_fl'), _aid('str_fr')
car_bid       = _bid('car1_chassis')
target_ang_v  = TARGET_V / WHEEL_RADIUS

print(f'[Model] ready  |  raycast wall sensing enabled  |  no track map in controller')

In [ ]:
# ============================================================
# Renderer & cameras
# ============================================================

renderer = mujoco.Renderer(model, height=480, width=640)

chase_cam              = mujoco.MjvCamera()
chase_cam.type         = mujoco.mjtCamera.mjCAMERA_TRACKING
chase_cam.trackbodyid  = car_bid
chase_cam.distance     = 12.0
chase_cam.elevation    = -20.0

overhead_cam             = mujoco.MjvCamera()
overhead_cam.type        = mujoco.mjtCamera.mjCAMERA_TRACKING
overhead_cam.trackbodyid = car_bid
overhead_cam.distance    = 30.0
overhead_cam.elevation   = -90.0
overhead_cam.azimuth     = 0.0

In [ ]:
# ============================================================
# Simulation Loop
# ============================================================

frames_chase    = []
frames_overhead = []
log_data        = []
os.makedirs("input_images", exist_ok=True)

print(f"Simulation start  (Qwen={'ON' if USE_QWEN else 'OFF'})")
print( "Control mode : VISION ONLY — no FF, no FB, no track geometry")

target_steer  = 0.0
current_steer = 0.0
stuck_since   = None

curve_action = "none"
drift_action = "centered"
curve_conf   = 0.0
drift_conf   = 0.0

for step in range(SIM_STEPS):
    t = step * SIM_DT
    if t >= MAX_SIM_TIME:
        print(f"\n⏱ Time limit {MAX_SIM_TIME}s reached.")
        break

    # ── State: position & heading ──────────────────────
    car_pos = data.xpos[car_bid, :2].copy()
    R3      = data.xmat[car_bid].reshape(3, 3)
    car_hdg = np.arctan2(R3[1, 0], R3[0, 0])
    chase_cam.azimuth = np.degrees(car_hdg)

    # ── Wall distances via raycasting (sensor only) ────────────
    geo_left, geo_right = get_wall_distances_raycast(
        model, data, car_pos, car_hdg, car_bid) 

    # ── Stuck detection ──────────────────────────────
    if geo_left < STUCK_DIST_THRESH or geo_right < STUCK_DIST_THRESH:
        if stuck_since is None:
            stuck_since = t
        elif t - stuck_since >= STUCK_WINDOW:
            print(f"\n🚫 Stuck for {STUCK_WINDOW}s → stopping")
            break
    else:
        stuck_since = None

    # ── Qwen judgment every JUDGMENT_INTERVAL steps ────────────
    if step % JUDGMENT_INTERVAL == 0:
        renderer.update_scene(data, camera=chase_cam)
        raw_img = Image.fromarray(renderer.render())

        hud_img = draw_hud(raw_img, geo_left, geo_right,
                           curve_action, drift_action,
                           curve_conf, drift_conf)
        hud_img.save(f"input_images/frame_{step:04d}.png")

        if USE_QWEN:
            resp = query_qwen(hud_img)
            curve_action, drift_action, curve_conf, drift_conf = \
                parse_response(resp)
        # (mock / ablation: keep last values unchanged)

        # ── ★ Vision-only steering target ───────────────────────
        target_steer = compute_steer_target_vision_only(
            curve_action, curve_conf,
            geo_left,     geo_right,
            target_steer, target_steer
        )

        print(f"[{t:.3f}s] "
              f"L={geo_left:.2f} R={geo_right:.2f} | "
              f"curve={curve_action:12s}({curve_conf:.2f}) "
              f"drift={drift_action:15s}({drift_conf:.2f}) "
              f"→ target={math.degrees(target_steer):+.1f}°")

        log_data.append({
            "Time":           t,
            "GeoLeft":        geo_left,
            "GeoRight":       geo_right,
            "CurveAction":    curve_action,
            "CurveConf":      curve_conf,
            "DriftAction":    drift_action,
            "DriftConf":      drift_conf,
            "TargetSteer_deg":  math.degrees(target_steer),
            "CurrentSteer_deg": math.degrees(current_steer),
        })

    # ── Emergency wall override ────────────────────────────────
    override_steer, override_active = geo_safety_override(geo_left, geo_right)
    if override_active:
        current_steer = override_steer
        if log_data:
            log_data[-1]["OverrideActive"] = True
        if step % JUDGMENT_INTERVAL == 0:
            print(f"  ⚡ EMERGENCY L={geo_left:.2f} R={geo_right:.2f} "
                  f"→ {math.degrees(override_steer):+.1f}°")
    else:
        current_steer += STEER_LERP * (target_steer - current_steer)
        current_steer  = float(np.clip(current_steer, -MAX_STEER_RAD, MAX_STEER_RAD))

    data.ctrl[a_drl] = target_ang_v
    data.ctrl[a_drr] = target_ang_v
    data.ctrl[a_sfl] = current_steer
    data.ctrl[a_sfr] = current_steer

    mujoco.mj_step(model, data)

    if step % RECORD_EVERY == 0:
        renderer.update_scene(data, camera=chase_cam)
        frames_chase.append(renderer.render())
        renderer.update_scene(data, camera=overhead_cam)
        frames_overhead.append(renderer.render())

    if is_wall_collision(model, data):
        print(f"  ★ Wall collision at {t:.3f}s")

    if step % 500 == 0:
        print(f"t={t:5.1f}s pos=({car_pos[0]:+6.1f},{car_pos[1]:+6.1f}) "
              f"L={geo_left:.2f} R={geo_right:.2f}")

In [ ]:
# ============================================================
# Save outputs
# ============================================================

imageio.mimsave("circuit_chase.gif",    frames_chase,    fps=FPS_OUTPUT, loop=0)
imageio.mimsave("circuit_overhead.gif", frames_overhead, fps=FPS_OUTPUT, loop=0)
print("Saved GIFs")

df = pd.DataFrame(log_data)
display(df)
df.to_csv('circuit_log.csv', index=False)

# ── Analysis plot ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].set_ylabel('Wall Dist (m)')
axes[0].plot(df['Time'], df['GeoLeft'],  'b--', label='Left  (raycast)')
axes[0].plot(df['Time'], df['GeoRight'], 'c--', label='Right (raycast)')
axes[0].axhline(SAFETY_DIST, color='r', ls=':', alpha=0.5, label='Safety')
axes[0].legend(fontsize=8); axes[0].grid(True)

axes[1].set_ylabel('Qwen Curve Conf')
axes[1].plot(df['Time'], df['CurveConf'], 'm-', label='curve confidence')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=8); axes[1].grid(True)

axes[2].set_ylabel('Steer Angle (deg)')
axes[2].set_xlabel('Time (s)')
axes[2].plot(df['Time'], df['TargetSteer_deg'],  'orange', ls='--', label='Target')
axes[2].plot(df['Time'], df['CurrentSteer_deg'], 'red',    ls='-',  label='Current')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].legend(fontsize=8); axes[2].grid(True)

plt.suptitle('Qwen2.5-VL Circuit: Vision-Only Steering (no FF, no FB)')
fig.tight_layout()
plt.savefig('circuit_analysis.png')
print("Saved circuit_analysis.png")

In [ ]:
from IPython.display import Image
Image(open("circuit_chase.gif", 'rb').read())

In [ ]:
Image(open("circuit_overhead.gif", 'rb').read())